# Import libraries

In [1]:
from Simulation.system_functions import PolymerCSTR
import numpy as np
import os
from utils.td3_helpers import load_and_prepare_system_data, filling_the_buffer, add_steady_state_samples, print_accuracy, ReplayDataset
import torch
from TD3Agent.agent import TD3Agent
from Simulation.mpc import MpcSolver
from torch.utils.data import DataLoader

## Initialize the system

In [2]:
# First initiate the system
# Parameters
Ad = 2.142e17           # h^-1
Ed = 14897              # K
Ap = 3.816e10           # L/(molh)
Ep = 3557               # K
At = 4.50e12            # L/(molh)
Et = 843                # K
fi = 0.6                # Coefficient
m_delta_H_r = -6.99e4   # j/mol
hA = 1.05e6             # j/(Kh)
rhocp = 1506            # j/(Kh)
rhoccpc = 4043          # j/(Kh)
Mm = 104.14             # g/mol
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

In [3]:
# Design Parameters
CIf = 0.5888    # mol/L
CMf = 8.6981    # mol/L
Qi = 108.       # L/h
Qs = 459.       # L/h
Tf = 330.       # K
Tcf = 295.      # K
V = 3000.       # L
Vc = 3312.4     # L

system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

In [4]:
# Steady State Inputs
Qm_ss = 378.    # L/h
Qc_ss = 471.6   # L/h

system_steady_state_inputs = np.array([Qc_ss, Qm_ss])

In [5]:
# Sampling time of the system
delta_t = 0.5 # 30 mins

In [6]:
# Initiate the CSTR for steady state values
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states={"ss_inputs":cstr.ss_inputs,
               "y_ss":cstr.y_ss}

## Loading the system matrices, min max scaling, and min max of the states

In [7]:
dir_path = os.path.join(os.getcwd(), "Data")

In [8]:
# Defining the range of setpoints for data generation
setpoint_y = np.array([[2.8, 320.],
                       [5., 326.]])
u_min = np.array([71.6, 78])
u_max = np.array([870, 670])

system_data = load_and_prepare_system_data(steady_states=steady_states, setpoint_y=setpoint_y, u_min=u_min, u_max=u_max)

In [9]:
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]

In [10]:
data_min = system_data["data_min"]
data_max = system_data["data_max"]

In [11]:
min_max_states = {'max_s': np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ]),
                  'min_s': np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])}

In [12]:
y_sp_scaled_deviation = system_data["y_sp_scaled_deviation"]

In [13]:
b_min = system_data["b_min"]
b_max = system_data["b_max"]

In [14]:
min_max_dict = system_data["min_max_dict"]
min_max_dict["x_max"] = np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ])
min_max_dict["x_min"] = np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])

## Setting The hyperparameters for the TD3 Agent

In [15]:
set_points_number = int(C_aug.shape[0])
inputs_number = int(B_aug.shape[1])
STATE_DIM = int(A_aug.shape[0]) + set_points_number + inputs_number
ACTION_DIM = int(B_aug.shape[1])
n_outputs = C_aug.shape[0]
ACTOR_LAYER_SIZES = [256] * 3
CRITIC_LAYER_SIZES = [256] * 3
BUFFER_CAPACITY = 5_000_000
ACTOR_LR = 1e-3
CRITIC_LR = 1e-3
SMOOTHING_STD = 0.000001
NOISE_CLIP = 0.000001
GAMMA = 0.99
TAU = 0.005 # 0.01
MAX_ACTION = 1
POLICY_DELAY = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 128
STD_START = 0.02
STD_END = 0.001
STD_DECAY_RATE = 0.99992
STD_DECAY_MODE = "exp"

In [16]:
td3_agent = TD3Agent(
    state_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    actor_hidden=ACTOR_LAYER_SIZES,
    critic_hidden=CRITIC_LAYER_SIZES,
    gamma=GAMMA,
    actor_lr=ACTOR_LR,
    critic_lr=CRITIC_LR,
    batch_size=BATCH_SIZE,
    policy_delay=POLICY_DELAY,
    target_policy_smoothing_noise_std=SMOOTHING_STD,
    noise_clip=NOISE_CLIP,
    max_action=MAX_ACTION,
    tau=TAU,
    std_start=STD_START,
    std_end=STD_END,
    std_decay_rate=STD_DECAY_RATE,
    std_decay_mode=STD_DECAY_MODE,
    buffer_size=BUFFER_CAPACITY,
    device=DEVICE,
    mode="mpc"
    )

# Filling the buffer

In [17]:
# MPC parameters
predict_h = 9
cont_h = 3
b1 = (b_min[0], b_max[0])
b2 = (b_min[1], b_max[1])
bnds = (b1, b2)*cont_h
cons = []
IC_opt = np.zeros(inputs_number*cont_h)
Q1 = 5
Q2 = 1
R1 = 1
R2 = 1
Q_rew = np.array([[5, 0], [0, 1]])
R_rew = np.array([[1, 0], [0, 1]])

In [18]:
MPC_obj = MpcSolver(A_aug, B_aug, C_aug,
    Q_out=np.array([Q1, Q2]),
    R_in=np.array([R1, R2]),
    NP=predict_h,
    NC=cont_h)

In [19]:
steady_states_samples_number = 100000
mpc_pretrain_samples_numbers = BUFFER_CAPACITY - steady_states_samples_number

In [20]:
filling_the_buffer(
        min_max_dict,
        A_aug, B_aug, C_aug,
        MPC_obj,
        mpc_pretrain_samples_numbers,
        Q_rew, R_rew,
        td3_agent,
        IC_opt, bnds, cons, chunk_size= 100_000)

Processing chunk 1/49 (size=100000)
Processing chunk 2/49 (size=100000)
Processing chunk 3/49 (size=100000)
Processing chunk 4/49 (size=100000)
Processing chunk 5/49 (size=100000)
Processing chunk 6/49 (size=100000)
Processing chunk 7/49 (size=100000)
Processing chunk 8/49 (size=100000)
Processing chunk 9/49 (size=100000)
Processing chunk 10/49 (size=100000)
Processing chunk 11/49 (size=100000)
Processing chunk 12/49 (size=100000)
Processing chunk 13/49 (size=100000)
Processing chunk 14/49 (size=100000)
Processing chunk 15/49 (size=100000)
Processing chunk 16/49 (size=100000)
Processing chunk 17/49 (size=100000)
Processing chunk 18/49 (size=100000)
Processing chunk 19/49 (size=100000)
Processing chunk 20/49 (size=100000)
Processing chunk 21/49 (size=100000)
Processing chunk 22/49 (size=100000)
Processing chunk 23/49 (size=100000)
Processing chunk 24/49 (size=100000)
Processing chunk 25/49 (size=100000)
Processing chunk 26/49 (size=100000)
Processing chunk 27/49 (size=100000)
Processing

In [21]:
add_steady_state_samples(
        min_max_dict,
        A_aug, B_aug, C_aug,
        MPC_obj,
        steady_states_samples_number,
        Q_rew, R_rew,
        td3_agent,
        IC_opt, bnds, cons, chunk_size= 100_000)

Processing chunk 1/1 (size=100000)
Replay buffer has been filled up with the steady_state values.


## Pre training the Agent

In [22]:
n = len(td3_agent.buffer)
s = torch.from_numpy(td3_agent.buffer.states[:n]).float()
a = torch.from_numpy(td3_agent.buffer.actions[:n]).float()
r = torch.from_numpy(td3_agent.buffer.rewards[:n]).float()
ns = torch.from_numpy(td3_agent.buffer.next_states[:n]).float()
d = torch.from_numpy(td3_agent.buffer.dones[:n]).float()

dataset = ReplayDataset(s, a, r, ns, d)
pin = DEVICE.type == "cuda"
data_loader = DataLoader(dataset, batch_size=8192, shuffle=True, num_workers=0, pin_memory=pin, drop_last=True)

In [23]:
td3_agent.pretrain_from_buffer(
    num_actor_epochs=1000,
    num_critic_epochs=500,
    data_loader=data_loader,
    use_target_noise_critic=True,
    log_interval=10,
)

[pretrain][actor_bc] ep=1 loss=5.7092e-03 lr=1.00e-03
[pretrain][actor_bc] ep=10 loss=5.0954e-05 lr=1.00e-03
[pretrain][actor_bc] ep=20 loss=1.6521e-05 lr=9.99e-04
[pretrain][actor_bc] ep=30 loss=8.7590e-06 lr=9.98e-04
[pretrain][actor_bc] ep=40 loss=6.0289e-06 lr=9.96e-04
[pretrain][actor_bc] ep=50 loss=4.3068e-06 lr=9.94e-04
[pretrain][actor_bc] ep=60 loss=3.8853e-06 lr=9.91e-04
[pretrain][actor_bc] ep=70 loss=3.1920e-06 lr=9.88e-04
[pretrain][actor_bc] ep=80 loss=2.5618e-06 lr=9.84e-04
[pretrain][actor_bc] ep=90 loss=2.3889e-06 lr=9.80e-04
[pretrain][actor_bc] ep=100 loss=1.9504e-06 lr=9.76e-04
[pretrain][actor_bc] ep=110 loss=1.9375e-06 lr=9.70e-04
[pretrain][actor_bc] ep=120 loss=1.7353e-06 lr=9.65e-04
[pretrain][actor_bc] ep=130 loss=1.5044e-06 lr=9.59e-04
[pretrain][actor_bc] ep=140 loss=1.4906e-06 lr=9.52e-04
[pretrain][actor_bc] ep=150 loss=1.4041e-06 lr=9.46e-04
[pretrain][actor_bc] ep=160 loss=1.4634e-06 lr=9.38e-04
[pretrain][actor_bc] ep=170 loss=1.2069e-06 lr=9.30e-04
[pr

RuntimeError: C:\actions-runner\_work\pytorch\pytorch\pytorch\build\aten\src\ATen\RegisterCPU_0.cpp:1820: SymIntArrayRef expected to contain only concrete integers

## Saving and loading the agent to make sure the agent has been stored

In [24]:
filename_agent = td3_agent.save(dir_path)

Saved TD3 checkpoint to: C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\Polymer_example\Data\td3_20260208_154119.pkl


## Checking the accuracy of the agent and compare it to the MPC actions

In [34]:
print_accuracy(td3_agent, n_samples=20, device=DEVICE)

Agent r2 score for the predicted inputs compare to MPC inputs: 0.999242
Agent r2 score for the predicted input 1 compare to MPC input 1: 0.999341
Agent r2 score for the predicted input 2 compare to MPC input 2: 0.999144
